# MODEL_EVALUATION — M3-01 / Qwen3-8B

This notebook evaluates **one candidate** on Colab, specifically Qwen3-8B, run on 2 other models will be test later. A100 (sm_80) results are preliminary: they do **not** certify Blackwell sm_120, RTX 5090 performance, Docker isolation, or the official six-hour limit.

Repository references:
- [TASKS M3-01](../docs/TASKS.md): serving, memory, tokens/s, JSON-schema output, repeatability.
- [SPEC §7.13 / §18](../docs/SPEC.md): local API, exact output capture, synchronous contracts.
- [Architecture proposal §8](../docs/ARCHITECTURE_PROPOSAL.md): candidate selection and later bake-off.
- [Implementation plan §5](../docs/IMPLEMENTATION_PLAN.md): environment fingerprints and reproducibility.


## 1. Configuration

Leave the repository link blank until you enter your own URL. For a private repository, configure Git authentication separately; do not put tokens in this notebook or the URL.

Default stack: vLLM **0.10.1.1**, isolated Python **3.12**, Transformers **4.55.2**. This is an exploratory A100 stack, not the pending M4-01 production lock. Exact resolved packages are exported for M4 review. Only change the stack deliberately and start a new run.

Default decoding is **greedy, thinking off**, matching M3-01. Qwen recommends other sampling settings for general quality; do not treat this test as a benchmark of its best possible reasoning quality.


In [4]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, subprocess, hashlib, time, csv, math, statistics
from concurrent.futures import ThreadPoolExecutor, as_completed

REPO_URL = "https://github.com/Kyoyooo/DataMate.git"  # REQUIRED: paste your Git repository URL here.
REPO_REF = ""  # Optional immutable commit/tag. Empty = repository default checkout.
REPO_DIR = Path("/content/DataMate")
EVAL_ROOT = Path("/content/datamate_evaluation")
MODEL_ID = "Qwen/Qwen3-8B"
MODEL_REVISION = ""  # Empty = resolve latest repository commit BEFORE the cutoff.
WEIGHT_CUTOFF = datetime.fromisoformat("2026-10-26T00:00:00+07:00")
VLLM_VERSION = "0.10.1.1"
TRANSFORMERS_VERSION = "4.55.2"
VENV_DIR = Path("/content/datamate_serving_py312")
SERVED_NAME = "datamate-eval"
PORT = 8000
BASE_URL = f"http://127.0.0.1:{PORT}"
SEED = 20261116
MAX_MODEL_LEN = 8192
MAX_NUM_SEQS = 8
TARGET_EXECUTOR_GIB = 28.8  # ~90% of a nominal 32 GiB target GPU.
CONCURRENCIES = [1, 4]  # Optional: append 8 after a successful first run.
REPEATS = 2
REQUEST_TIMEOUT_S = 180
STARTUP_TIMEOUT_S = 900
EXECUTE_FIXTURE_SQL = True  # Existing read-only executor; timer interrupt, NOT process isolation.
SQL_TIMEOUT_S = 2.0
LONG_CONTEXT_FRACTION = 0.75
# Fields below are manual compliance evidence, not automatically inferred from the model name.
LICENSE_REVIEWED_AND_PERMITTED = False  # Set True only after reviewing the linked license.
RELEASE_EVIDENCE_URL = ""  # Fill with evidence that original weights were public before cutoff.
RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR = EVAL_ROOT / RUN_STAMP
RUN_DIR.mkdir(parents=True, exist_ok=False)

def save_json(name, value):
    path = RUN_DIR / name
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8")
    return path

def run_command(args, **kwargs):
    return subprocess.run([str(x) for x in args], check=True, text=True, **kwargs)

print("Run directory:", RUN_DIR)
assert REPEATS >= 2 and max(CONCURRENCIES) <= MAX_NUM_SEQS
assert MODEL_ID == "Qwen/Qwen3-8B", "This notebook is tested structurally for Qwen3-8B only; adapt candidate-specific settings explicitly."


Run directory: /content/datamate_evaluation/20261109T040332_785805Z


## 2. Clone and pin the project

An existing checkout is reused without pulling, resetting, or overwriting local changes. Check its origin/ref manually if reused. A dirty checkout stops the experiment so the recorded commit identifies the source unambiguously.


In [5]:
if not REPO_URL.strip():
    raise ValueError("Fill REPO_URL in section 1, then rerun from here.")
if not REPO_DIR.exists():
    run_command(["git", "clone", "--", REPO_URL, REPO_DIR])
elif not (REPO_DIR / ".git").is_dir():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout.")
if REPO_REF:
    run_command(["git", "-C", REPO_DIR, "checkout", "--detach", REPO_REF])
dirty = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True
).strip()
if dirty:
    raise RuntimeError("Checkout has local changes. Use a clean checkout or a separate REPO_DIR.")
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
for relative in ["src/datamate/contracts/types.py", "src/datamate/prompts/templates.py",
                 "tests/fixtures/split_mini/questions.jsonl"]:
    assert (REPO_DIR / relative).is_file(), relative
print("Repository commit:", REPO_COMMIT)


Repository commit: d24001107dec9a3c431f987d55a15f09822420d5


## 3. Short GPU check

Use `nvidia-smi` before installing/loading CUDA libraries in the notebook kernel. The server will query CUDA compute capability from its isolated environment later. An A100 is valid for preliminary API/accuracy experiments, not sm_120 acceptance.


In [6]:
GPU_QUERY = ["nvidia-smi", "--query-gpu=index,name,uuid,memory.total,memory.used,driver_version",
             "--format=csv,noheader,nounits"]

def gpu_snapshot():
    raw = subprocess.check_output(GPU_QUERY, text=True)
    result = []
    for row in csv.reader(raw.splitlines()):
        index, name, uuid, total, used, driver = [x.strip() for x in row]
        result.append(dict(index=int(index), name=name, uuid=uuid,
                           total_mib=float(total), used_mib=float(used), driver=driver))
    return result

GPUS = gpu_snapshot()
assert GPUS, "No NVIDIA GPU. Select an A100 GPU runtime."
print(json.dumps(GPUS, indent=2))
print("Notebook Python:", sys.version)
if "A100" not in GPUS[0]["name"]:
    print("NOTICE: This is not an A100; retain the actual GPU identity in the report.")
save_json("gpu_before_setup.json", GPUS)


[
  {
    "index": 0,
    "name": "NVIDIA A100-SXM4-40GB",
    "uuid": "GPU-442c4c9b-13e5-c052-f64e-19a778bef1d8",
    "total_mib": 40960.0,
    "used_mib": 0.0,
    "driver": "580.82.07"
  }
]
Notebook Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


PosixPath('/content/datamate_evaluation/20261109T040332_785805Z/gpu_before_setup.json')

## 4. Install dependencies

The notebook kernel installs only lightweight evaluation/repository dependencies. vLLM and its matching PyTorch are installed in a **separate Python 3.12 virtual environment**, avoiding replacement of Colab's CUDA stack. No model is loaded in the notebook kernel.

Do not install the repository's historical CPU lock over the serving stack. These exploratory versions satisfy the current package constraints; give the exported lock to M4 before using the numbers for a release decision. If a package version fails to resolve, stop and record the failure; do not silently upgrade everything.

Sources: [vLLM 0.10.1.1 GPU installation](https://docs.vllm.ai/en/v0.10.1.1/getting_started/installation/gpu.html), [Qwen3-8B model card](https://huggingface.co/Qwen/Qwen3-8B).


In [7]:
run_command([sys.executable, "-m", "pip", "install", "--quiet",
             "uv==0.8.15", "duckdb==1.3.2", "pydantic==2.11.7",
             "requests==2.32.5", "huggingface-hub==0.34.4", "jsonschema==4.25.1"])
import shutil
UV = shutil.which("uv")
assert UV, "uv executable not found after installation."
if not VENV_DIR.exists():
    run_command([UV, "venv", "--python", "3.12", "--seed", VENV_DIR])
SERVER_PYTHON = VENV_DIR / "bin/python"
run_command([UV, "pip", "install", "--python", SERVER_PYTHON,
             f"vllm=={VLLM_VERSION}", f"transformers=={TRANSFORMERS_VERSION}"])
run_command([SERVER_PYTHON, "-m", "pip", "check"])
sys.path.insert(0, str(REPO_DIR / "src"))
import requests
from huggingface_hub import HfApi, snapshot_download
from IPython.display import display, Markdown

kernel_freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
server_freeze = subprocess.check_output([str(SERVER_PYTHON), "-m", "pip", "freeze"], text=True)
(RUN_DIR / "kernel_requirements.txt").write_text(kernel_freeze, encoding="utf-8")
(RUN_DIR / "serving_requirements.txt").write_text(server_freeze, encoding="utf-8")
server_probe = subprocess.check_output([str(SERVER_PYTHON), "-c",
    "import torch,json,sys,vllm,transformers; print(json.dumps(dict("
    "python=sys.version,torch=torch.__version__,cuda=torch.version.cuda,"
    "vllm=vllm.__version__,transformers=transformers.__version__,"
    "gpu=torch.cuda.get_device_name(0),capability=torch.cuda.get_device_capability(0))))"
], text=True)
SERVER_ENV = json.loads(server_probe.strip().splitlines()[-1])
save_json("serving_environment.json", SERVER_ENV)
print(json.dumps(SERVER_ENV, indent=2))


{
  "python": "3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]",
  "torch": "2.7.1+cu126",
  "cuda": "12.6",
  "vllm": "0.10.1.1",
  "transformers": "4.55.2",
  "gpu": "NVIDIA A100-SXM4-40GB",
  "capability": [
    8,
    0
  ]
}


## 5. Resolve, download and inspect the checkpoint

The default resolves a historical commit before **2026-10-26 00:00 GMT+7**. This prevents a later revision silently entering the run. A pre-cutoff repository commit is supporting evidence, not a complete organizer eligibility ruling; review the original weights and license.

Only download the safetensors checkpoint, config, tokenizer and model-card/license files. Parameter count is computed from **actual tensor shapes in safetensors headers**, without allocating GPU memory. Qwen3-8B is an unquantized dense checkpoint; this stored-tensor count must not be reused as a universal architecture/MoE/quantized parameter counter.

Sources: [official model repository](https://huggingface.co/Qwen/Qwen3-8B), [license](https://huggingface.co/Qwen/Qwen3-8B/blob/main/LICENSE), [Qwen3 release announcement](https://qwenlm.github.io/blog/qwen3/).


In [8]:
api = HfApi()
commits = list(api.list_repo_commits(MODEL_ID))
eligible_commits = [c for c in commits if c.created_at < WEIGHT_CUTOFF]
if not eligible_commits:
    raise RuntimeError("No pre-cutoff repository commit found.")
if MODEL_REVISION:
    info = api.model_info(MODEL_ID, revision=MODEL_REVISION)
    revision = info.sha
    commit = next((c for c in commits if c.commit_id == revision), None)
    if commit is None or commit.created_at >= WEIGHT_CUTOFF:
        raise RuntimeError("Selected revision could not be verified as pre-cutoff.")
else:
    commit = max(eligible_commits, key=lambda c: c.created_at)
    revision = commit.commit_id
info = api.model_info(MODEL_ID, revision=revision)
card = info.card_data.to_dict() if info.card_data else {}
license_id = card.get("license")
MODEL_PATH = Path(snapshot_download(
    repo_id=MODEL_ID, revision=revision,
    allow_patterns=["*.safetensors", "*.json", "*.txt", "*.model", "*.tiktoken",
                    "*.jinja", "*.jinja2", "README.md", "LICENSE*"],
))
import struct
tensor_count = 0
tensor_dtypes = set()
weight_files = sorted(MODEL_PATH.glob("*.safetensors"))
assert weight_files, "No safetensors weights at the selected revision."
for weight_file in weight_files:
    with weight_file.open("rb") as fh:
        size = struct.unpack("<Q", fh.read(8))[0]
        assert size < 100_000_000, "Unexpected safetensors header size."
        header = json.loads(fh.read(size))
    for name, tensor in header.items():
        if name == "__metadata__":
            continue
        tensor_count += math.prod(tensor["shape"])
        tensor_dtypes.add(tensor["dtype"])
assert tensor_dtypes <= {"BF16", "F16", "F32"}, "Review count methodology for quantized/other tensors."
weight_hashes = {}
for weight_file in weight_files:
    digest = hashlib.sha256()
    with weight_file.open("rb") as fh:
        for block in iter(lambda: fh.read(16 * 1024 * 1024), b""):
            digest.update(block)
    weight_hashes[weight_file.name] = digest.hexdigest()
MODEL_EVIDENCE = dict(
    model_id=MODEL_ID, revision=revision, revision_date=commit.created_at.isoformat(),
    cutoff=WEIGHT_CUTOFF.isoformat(), license=license_id,
    license_reviewed_and_permitted=LICENSE_REVIEWED_AND_PERMITTED,
    original_release_evidence=RELEASE_EVIDENCE_URL,
    stored_tensor_parameters=tensor_count, tensor_dtypes=sorted(tensor_dtypes),
    parameter_limit_pass=tensor_count <= 15_000_000_000,
    weights_sha256=weight_hashes, checkpoint_url=f"https://huggingface.co/{MODEL_ID}/tree/{revision}",
)
save_json("model_evidence.json", MODEL_EVIDENCE)
print(json.dumps({k: v for k, v in MODEL_EVIDENCE.items() if k != "weights_sha256"}, indent=2))
assert MODEL_EVIDENCE["parameter_limit_pass"], "Checkpoint exceeds the parameter budget."


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

... (7 outputs left out)

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

{
  "model_id": "Qwen/Qwen3-8B",
  "revision": "b968826d9c46dd6066d109eabc6255188de91218",
  "revision_date": "2025-07-26T03:49:13+00:00",
  "cutoff": "2026-10-26T00:00:00+07:00",
  "license": "apache-2.0",
  "license_reviewed_and_permitted": false,
  "original_release_evidence": "",
  "stored_tensor_parameters": 8190735360,
  "tensor_dtypes": [
    "BF16"
  ],
  "parameter_limit_pass": true,
  "checkpoint_url": "https://huggingface.co/Qwen/Qwen3-8B/tree/b968826d9c46dd6066d109eabc6255188de91218"
}


## 6. Reuse repository contracts, prompts and synthetic fixtures

The fixture contains hand-derived expected answers. Its scripted responses supply **reference frames only** for isolated solver tests; generated SQL always comes from the real candidate. This is not a training/held-out benchmark.

Parse requests use `parse.frame.v3` and the generated schema. Solver requests use reference frames/windows so parser errors do not obscure solver compatibility. Added temporal questions exercise the new contract forms. A deliberately failing SQL statement exercises repair.

No public/private-test data is used. An actual PA bake-off needs the Lead's business-fold harness (M3-08/L-12).


In [9]:
from datamate.contracts import CONTRACTS_VERSION, Question, QuestionFrame, SolverOutput
from datamate.contracts import Decision, SolverAttempt
from datamate.config import RunConfig, ExecLimits
from datamate.catalog import setup_business
from datamate.rules import load_split
from datamate.temporal import resolve_windows
from datamate.prompts import render_parse, render_solve, render_repair, template_sha256
from datamate.lexicon import fold
from datamate.decision import pre_solve_decision
from datamate.solver import check_sql, to_typed_value
from datamate.sandbox import run_sql
from datamate.formatting import to_submission
from datamate.eval.score import score_one
from datetime import date

SPLIT_DIR = REPO_DIR / "tests/fixtures/split_mini"
split = load_split(SPLIT_DIR)
questions_raw = [json.loads(x) for x in (SPLIT_DIR / "questions.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
references = {x["id"]: x for x in [json.loads(s) for s in
    (SPLIT_DIR / "answers.jsonl").read_text(encoding="utf-8").splitlines() if s.strip()]}
script_entries = [json.loads(s) for s in (REPO_DIR / "tests/fixtures/scripted/split_mini.jsonl").read_text(
    encoding="utf-8").splitlines() if s.strip()]
reference_frames = {x["question"]: QuestionFrame.model_validate_json(x["content"])
                    for x in script_entries if x["stage"] == "parse"}
catalogs = {}
cases = []
cfg = RunConfig()
for line_no, raw in enumerate(questions_raw, 1):
    q = Question(**{**raw, "as_of_date": date.fromisoformat(raw["as_of_date"]), "line_no": line_no})
    if q.business_id not in catalogs:
        catalogs[q.business_id] = setup_business(
            SPLIT_DIR / "data" / q.business_id, q.business_id, RUN_DIR / "work")
    cat = catalogs[q.business_id]
    cases.append(dict(case_id=f"parse:{q.id}", stage="parse", template_id="parse.frame.v3",
        messages=render_parse(q, split, cat), schema=QuestionFrame.model_json_schema(),
        max_tokens=1024, question=q, reference=references[q.id], diagnostic=False))
    frame = reference_frames[q.question]
    windows, error = resolve_windows(frame.time, q.as_of_date, split.temporal, cat.data_range)
    if not error and pre_solve_decision(frame, windows, cat, cfg) is None:
        cases.append(dict(case_id=f"solve:{q.id}", stage="solve", template_id="solve.sql.v1",
            messages=render_solve(q, frame, windows, split, cat), schema=SolverOutput.model_json_schema(),
            max_tokens=2048, question=q, frame=frame, reference=references[q.id], diagnostic=False))

cat = next(iter(catalogs.values()))
temporal_texts = [
    "Từ 1/3 đến 4/3 có nhiều đơn hơn cùng kỳ tháng trước không?",
    "Doanh thu 2 tháng gần nhất là bao nhiêu?",
    "Doanh thu tháng này so với cùng kỳ năm ngoái thay đổi bao nhiêu phần trăm?",
]
for i, text in enumerate(temporal_texts):
    q = Question(id=f"temporal_{i}", business_id=cat.business_id, question=text,
                 as_of_date=date(2026, 3, 4), line_no=100 + i)
    cases.append(dict(case_id=f"parse:temporal_{i}", stage="parse", template_id="parse.frame.v3",
        messages=render_parse(q, split, cat), schema=QuestionFrame.model_json_schema(),
        max_tokens=1024, question=q, diagnostic=True))
first_solve = next(c for c in cases if c["stage"] == "solve")
bad_sql = json.dumps({"sql": "SELECT deliberately_missing_column FROM ban_hang", "notes": ""})
cases.append({**first_solve, "case_id": "repair:invalid_column", "stage": "repair",
    "template_id": "repair.sql.v1", "messages": render_repair(first_solve["messages"], bad_sql,
    "E_SQL_EXEC", "Referenced column deliberately_missing_column was not found.")})
print("Fixture cases:", len(cases), "| contracts:", CONTRACTS_VERSION)
print("Templates:", template_sha256())


Fixture cases: 28 | contracts: 1.3.0
Templates: {'parse.frame.v3': '83fd39fe54c0b51eca2c1b63946fcfc7094957d2e6f26975d3965abbda03f359', 'repair.sql.v1': '369c4671f61c8fa538895b4d0167c6471837aecc032a689026c621f905a50051', 'rules.temporal.v1': '2fa07419a6e62d9b5e57e7d14495d0dcf28ca4a3e1d261c640cf2ac8f5484273', 'solve.sql.v1': 'ca4c1801325b71349a6aebf0834f0247a519f149a198f7234c571c1ab741372e'}


## 7. Count tokens and add a long-context diagnostic

The serving environment loads the local tokenizer on CPU. Prompt counts include the chat template with thinking disabled. The long-context probe expands a copied README section; the real rulebook is preserved. It checks context handling, not answer quality.

All requests must leave room for their output budget within `MAX_MODEL_LEN`. Overflow stops before serving; it is not silently truncated.


In [10]:
import copy
tokenizer_program = """
import sys,json
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained(sys.argv[1],local_files_only=True)
messages=json.load(sys.stdin)
print(json.dumps([len(tokenizer.apply_chat_template(m,tokenize=True,
    add_generation_prompt=True,enable_thinking=False)) for m in messages]))
"""
def token_counts(message_lists):
    output = subprocess.check_output([str(SERVER_PYTHON), "-c", tokenizer_program, str(MODEL_PATH)],
                                     input=json.dumps(message_lists, ensure_ascii=False), text=True)
    return json.loads(output.strip().splitlines()[-1])

long_case = copy.deepcopy(first_solve)
long_case.update(case_id="solve:long_context_diagnostic", diagnostic=True)
long_case.pop("reference", None)
system_text = long_case["messages"][0]["content"]
target_input = min(int(MAX_MODEL_LEN * LONG_CONTEXT_FRACTION),
                   MAX_MODEL_LEN - long_case["max_tokens"] - 64)
padding = "\nNOTE: Synthetic catalog context padding for serving tests only."
lo, hi = 0, 3000
while lo < hi:
    mid = (lo + hi + 1) // 2
    trial = copy.deepcopy(long_case["messages"])
    trial[0]["content"] = system_text.replace("\nDATA SCHEMA:", padding * mid + "\nDATA SCHEMA:", 1)
    if token_counts([trial])[0] <= target_input:
        lo = mid
    else:
        hi = mid - 1
long_case["messages"][0]["content"] = system_text.replace("\nDATA SCHEMA:", padding * lo + "\nDATA SCHEMA:", 1)
cases.append(long_case)
for case, n in zip(cases, token_counts([c["messages"] for c in cases]), strict=True):
    case["input_tokens_estimate"] = n
    assert n + case["max_tokens"] <= MAX_MODEL_LEN, (case["case_id"], n)
save_json("request_cases.json", cases)
print("Requests:", len(cases), "| maximum estimated input tokens:",
      max(c["input_tokens_estimate"] for c in cases))


Requests: 29 | maximum estimated input tokens: 6073


## 8. Launch one local vLLM replica and warm up

The server sees local weights only, with Hugging Face offline settings and telemetry disabled. This is **not an OS-level network isolation test**. Internet must remain available during setup; no inference API calls leave the Colab runtime.

The memory fraction is computed from the selected GPU's actual capacity. Peak GPU memory is sampled separately and includes CUDA overhead. A100 measurements cannot certify RTX 5090 performance.

Source: [vLLM engine arguments](https://docs.vllm.ai/en/v0.10.1.1/configuration/engine_args.html).


In [11]:
import threading, signal
GPU_UTILIZATION = min(0.90, TARGET_EXECUTOR_GIB * 1024 / GPUS[0]["total_mib"])
assert 0 < GPU_UTILIZATION < 1
if "server" in globals() and server.poll() is None:
    raise RuntimeError("A server from this notebook is already running. Run the cleanup cell first.")
with requests.Session() as session:
    session.trust_env = False
    try:
        session.get(BASE_URL + "/health", timeout=2)
    except requests.RequestException:
        pass
    else:
        raise RuntimeError(f"Port {PORT} already has a service; stop it or choose a new port.")
server_env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "HF_HUB_OFFLINE": "1",
              "TRANSFORMERS_OFFLINE": "1", "HF_HUB_DISABLE_TELEMETRY": "1",
              "VLLM_NO_USAGE_STATS": "1", "TOKENIZERS_PARALLELISM": "false"}
server_args = [
    str(SERVER_PYTHON), "-m", "vllm.entrypoints.openai.api_server",
    "--model", str(MODEL_PATH), "--tokenizer", str(MODEL_PATH),
    "--served-model-name", SERVED_NAME, "--host", "127.0.0.1", "--port", str(PORT),
    "--dtype", "bfloat16", "--max-model-len", str(MAX_MODEL_LEN),
    "--max-num-seqs", str(MAX_NUM_SEQS), "--tensor-parallel-size", "1",
    "--gpu-memory-utilization", str(GPU_UTILIZATION), "--seed", str(SEED),
    "--generation-config", "vllm", "--reasoning-parser", "qwen3",
    "--enable-prefix-caching", "--cpu-offload-gb", "0",
]
save_json("server_command.json", server_args)
gpu_samples = []
monitor_stop = threading.Event()
def monitor_gpu():
    while not monitor_stop.is_set():
        try:
            gpu_samples.append({"time": time.time(), **gpu_snapshot()[0]})
        except Exception as exc:
            gpu_samples.append({"time": time.time(), "monitor_error": str(exc)})
        monitor_stop.wait(0.5)
monitor_thread = threading.Thread(target=monitor_gpu, daemon=True)
monitor_thread.start()
server_log_handle = (RUN_DIR / "server.log").open("w", encoding="utf-8")
load_start = time.perf_counter()
server = subprocess.Popen(server_args, env=server_env, stdout=server_log_handle,
                          stderr=subprocess.STDOUT, start_new_session=True)
try:
    with requests.Session() as session:
        session.trust_env = False
        while True:
            if server.poll() is not None:
                raise RuntimeError("vLLM exited; inspect server.log.")
            try:
                if session.get(BASE_URL + "/health", timeout=2).status_code == 200:
                    break
            except requests.RequestException:
                pass
            if time.perf_counter() - load_start > STARTUP_TIMEOUT_S:
                raise TimeoutError("vLLM startup exceeded the configured timeout.")
            time.sleep(2)
    LOAD_SECONDS = time.perf_counter() - load_start
    print(f"Healthy after {LOAD_SECONDS:.1f}s; engine memory fraction={GPU_UTILIZATION:.4f}")
except Exception:
    os.killpg(server.pid, signal.SIGTERM)
    monitor_stop.set()
    server_log_handle.flush()
    print((RUN_DIR / "server.log").read_text(encoding="utf-8")[-6000:])
    raise


Healthy after 120.1s; engine memory fraction=0.7200


## 9. Evaluation criteria and exact HTTP capture

| Criterion | Check | Interpretation |
|---|---|---|
| Request compatibility | HTTP success, content present, finish reason = stop | API/decoding smoke test |
| Schema | Validate actual response with repository Pydantic models | JSON correctness is not answer correctness |
| Parser invariants | Original time spans exist after fold | Same check as the current parser |
| SQL | Repository check_sql + optional read-only execution/type conversion | Isolated fixture functional check |
| Repeatability | Exact content + reasoning match, two fixed-order passes per concurrency | Report failures, never silently average them away |
| Memory | Sampled peak VRAM; target engine budget fixed | A100 proxy, not sm_120 acceptance |
| Performance | p50/p95, aggregate completion tokens/s and requests/s | Warm-cache workload; exclude schema warm-up |

No automatic request retries are used here, so transport/model failures remain visible. Every HTTP body, request hash and response text is retained. No synthetic reasoning is generated.

Source: [JSON-schema responses in vLLM 0.10.1.1](https://docs.vllm.ai/en/v0.10.1.1/features/structured_outputs.html).


### 9a. Schema compatibility preflight (no GPU calls)

The pinned serving backend may reject Pydantic's `oneOf` in `TimeRef.expr` before generation. This adapter changes only a copied transport schema: every time-expression branch must have a unique string `kind.const`, `kind` becomes required, and that union uses `anyOf`. Required distinct tags make the branches mutually exclusive. The original `QuestionFrame` contract, source files, and response validation remain unchanged.

The adapter fails explicitly for unsupported branches, overlapping tags, or other `oneOf` keywords. It does not relax arbitrary unions, turn off structured generation, or bypass warm-up. Both schemas, their hashes, and the adapter policy are saved to `schema_evidence.json`. `strict=True` remains enabled. A local JSON Schema check is a preflight, not proof that the GPU backend will compile every construct; warm-up remains required.

If updating an already-running Colab session, run this new cell, then rerun the entire request/evaluation cell below to redefine `request_body` and repeat warm-up. The serving process does not need restarting for this request-only change. Use a new run if benchmark results were already collected; do not mix pre/post-adapter measurements.


In [12]:
import copy
from jsonschema import Draft202012Validator

SCHEMA_ADAPTER_POLICY = "timeref-kind-required-oneof-to-anyof.v1"

def schema_sha256(schema):
    canonical = json.dumps(schema, sort_keys=True, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

def schema_for_vllm(original):
    """Adapt only TimeRef.expr; application validation still uses the original model."""
    wire = copy.deepcopy(original)
    definitions = wire.get("$defs", {})
    expr = definitions.get("TimeRef", {}).get("properties", {}).get("expr", {})
    if "oneOf" in expr:
        if expr.get("discriminator", {}).get("propertyName") != "kind":
            raise ValueError("Unexpected TimeRef discriminator; review the schema adapter.")
        branches = expr["oneOf"]
        if not isinstance(branches, list) or not branches:
            raise ValueError("TimeRef.expr.oneOf must be a non-empty list.")
        seen_kinds = set()
        for branch in branches:
            ref = branch.get("$ref", "")
            if set(branch) != {"$ref"} or not ref.startswith("#/$defs/"):
                raise ValueError(f"Unsupported TimeRef branch: {branch}")
            definition = definitions[ref.removeprefix("#/$defs/")]
            kind = definition.get("properties", {}).get("kind", {})
            tag = kind.get("const")
            if definition.get("type") != "object" or kind.get("type") != "string":
                raise ValueError(f"Expected an object with a string kind: {ref}")
            if not isinstance(tag, str) or tag in seen_kinds:
                raise ValueError(f"Missing or overlapping kind.const: {ref}")
            seen_kinds.add(tag)
            # Pydantic's tagged union needs kind even when a variant has a default.
            required = definition.setdefault("required", [])
            if "kind" not in required:
                required.append("kind")
        # Required, distinct kind.const values make the branches mutually exclusive.
        expr["anyOf"] = expr.pop("oneOf")
        expr.pop("discriminator", None)

    def reject_remaining_oneof(node, location="$"):
        # Walk schema-valued keywords only; a property named 'oneOf' is not a keyword.
        if isinstance(node, dict):
            if "oneOf" in node:
                raise ValueError(f"Unsupported oneOf remains at {location}; review explicitly.")
            for key in ("$defs", "definitions", "properties", "patternProperties", "dependentSchemas"):
                for name, child in node.get(key, {}).items():
                    reject_remaining_oneof(child, f"{location}/{key}/{name}")
            for key in ("items", "additionalProperties", "contains", "not", "if", "then", "else", "propertyNames"):
                child = node.get(key)
                if isinstance(child, dict):
                    reject_remaining_oneof(child, f"{location}/{key}")
            for key in ("anyOf", "allOf", "prefixItems"):
                for index, child in enumerate(node.get(key, [])):
                    reject_remaining_oneof(child, f"{location}/{key}/{index}")
    reject_remaining_oneof(wire)
    Draft202012Validator.check_schema(wire)
    return wire

schema_evidence = {"policy": SCHEMA_ADAPTER_POLICY, "templates": {}}
for case in cases:
    original = case["schema"]
    serving = schema_for_vllm(original)
    case["serving_schema"] = serving
    evidence = {
        "original_sha256": schema_sha256(original),
        "serving_sha256": schema_sha256(serving),
        "changed": original != serving,
        "original_schema": original,
        "serving_schema": serving,
    }
    previous = schema_evidence["templates"].get(case["template_id"])
    if previous is not None and previous != evidence:
        raise ValueError("A template ID has conflicting schemas in this run.")
    schema_evidence["templates"][case["template_id"]] = evidence
save_json("schema_evidence.json", schema_evidence)
for template_id, evidence in schema_evidence["templates"].items():
    print(template_id, "adapted:", evidence["changed"],
          "serving schema SHA-256:", evidence["serving_sha256"])


parse.frame.v3 adapted: True serving schema SHA-256: a42a3bd15e00791dd88e9246792a94401d8dcf371cef61984f93a8e2dd76f0d8
solve.sql.v1 adapted: False serving schema SHA-256: 907e621e5a6b0d9b9048b9c023944ad2a3da2c8d73ccfc1a8ee9a7f0d2387889
repair.sql.v1 adapted: False serving schema SHA-256: 907e621e5a6b0d9b9048b9c023944ad2a3da2c8d73ccfc1a8ee9a7f0d2387889


In [13]:
def request_body(case):
    return {
        "model": SERVED_NAME, "messages": case["messages"],
        "temperature": 0.0, "top_p": 1.0, "top_k": -1, "min_p": 0.0,
        "presence_penalty": 0.0, "frequency_penalty": 0.0, "repetition_penalty": 1.0,
        "seed": SEED, "n": 1, "max_tokens": case["max_tokens"],
        "stream": False, "chat_template_kwargs": {"enable_thinking": False},
        "response_format": {"type": "json_schema", "json_schema": {
            "name": case["template_id"].replace(".", "_"),
            "schema": case["serving_schema"], "strict": True}},
    }

def evaluate_response(case, body, status, raw_text, elapsed, error=None):
    record = dict(case_id=case["case_id"], stage=case["stage"], diagnostic=case["diagnostic"],
        request_sha256=hashlib.sha256(json.dumps(body, sort_keys=True, ensure_ascii=False,
            separators=(",", ":")).encode()).hexdigest(),
        request=body, schema_adapter=SCHEMA_ADAPTER_POLICY,
        original_schema_sha256=schema_sha256(case["schema"]),
        serving_schema_sha256=schema_sha256(case["serving_schema"]),
        http_status=status, raw_response_text=raw_text,
        elapsed_s=elapsed, error=error, transport_ok=False, schema_valid=False,
        app_valid=False, sql_ok=None, typed_ok=None, fixture_score=None,
        content=None, reasoning=None, finish_reason=None,
        prompt_tokens=None, completion_tokens=None)
    if error is not None or status != 200:
        return record
    try:
        payload = json.loads(raw_text)
        choice = payload["choices"][0]
        message = choice["message"]
        record.update(content=message.get("content"),
            reasoning=message.get("reasoning_content", message.get("reasoning")),
            finish_reason=choice.get("finish_reason"),
            prompt_tokens=payload.get("usage", {}).get("prompt_tokens"),
            completion_tokens=payload.get("usage", {}).get("completion_tokens"))
        record["transport_ok"] = isinstance(record["content"], str)
        model = QuestionFrame if case["stage"] == "parse" else SolverOutput
        parsed = model.model_validate_json(record["content"])
        record["schema_valid"] = True
        record["app_valid"] = True
        if case["stage"] == "parse":
            record["app_valid"] = all(fold(ref.span) in fold(case["question"].question)
                                      for ref in parsed.time)
            if "reference" in case and case["reference"]["type"] not in ("unanswerable", "abstain"):
                record["expected_type_correct"] = parsed.expected_type == case["reference"]["type"]
        else:
            sql_error, sql_detail = check_sql(parsed.sql)
            record.update(sql_ok=sql_error is None, sql_error=sql_error, sql_detail=sql_detail)
            if EXECUTE_FIXTURE_SQL and sql_error is None:
                cat = catalogs[case["question"].business_id]
                execution = run_sql(cat.db_path, parsed.sql, ExecLimits(
                    sql_timeout_s=SQL_TIMEOUT_S, max_result_rows=1000, memory_limit_mb=2048))
                record["execution"] = execution.model_dump(mode="json")
                value, type_error = to_typed_value(case["frame"], execution) if execution.ok else (None, execution.error_code)
                record.update(typed_ok=execution.ok and type_error is None, type_error=type_error)
                if value is not None and "reference" in case and not case["diagnostic"]:
                    decision = Decision(kind="answer", value=value, reason_code="D_ANSWER")
                    submission = to_submission(case["question"].id, decision,
                                               case["frame"].ranking, cfg.formatting)
                    record["fixture_score"] = score_one(submission.to_wire(), case["reference"])
    except Exception as exc:
        record["validation_error"] = f"{type(exc).__name__}: {exc}"
    return record

def call_case(case):
    body = request_body(case)
    started = time.perf_counter()
    try:
        with requests.Session() as session:
            session.trust_env = False
            response = session.post(BASE_URL + "/v1/chat/completions", json=body,
                                    timeout=REQUEST_TIMEOUT_S)
        elapsed = time.perf_counter() - started
        return evaluate_response(case, body, response.status_code, response.text, elapsed)
    except requests.RequestException as exc:
        return evaluate_response(case, body, None, None, time.perf_counter() - started,
                                 f"{type(exc).__name__}: {exc}")

# Warm the actual parse and solver schema paths. Save cold latency separately.
warmups = [call_case(next(c for c in cases if c["stage"] == stage))
           for stage in ("parse", "solve", "repair")]
save_json("warmups.json", warmups)
for result in warmups:
    print(result["stage"], "HTTP", result["http_status"],
          "schema_valid", result["schema_valid"], "seconds", round(result["elapsed_s"], 2))
failed_warmups = [r for r in warmups if not (
    r["transport_ok"] and r["schema_valid"] and r["finish_reason"] == "stop")]
if failed_warmups:
    for result in failed_warmups:
        print("FAILED:", result["stage"], "HTTP:", result["http_status"],
              "finish_reason:", result["finish_reason"],
              "error:", result.get("error"),
              "validation_error:", result.get("validation_error"))
        print("Raw response:", (result.get("raw_response_text") or "")[:2000])
    raise RuntimeError("Warm-up failed. Inspect warmups.json, schema_evidence.json and server.log before benchmarking.")


parse HTTP 200 schema_valid True seconds 3.69
solve HTTP 200 schema_valid True seconds 2.01
repair HTTP 200 schema_valid True seconds 1.71


## 10. Run two passes at each concurrency

The order of submission is fixed. GPU batching may still vary; exact mismatches are reported. Local fixture execution happens after HTTP timing and is excluded from measured API latency. Batch wall time includes that validation work; its throughput is an observed evaluator rate, not a pure server maximum.

Keep this workload unchanged when comparing candidates. Prefix caching is enabled; these are warm-cache figures, not estimates for arbitrary business schemas.


In [14]:
records = []
batch_metrics = []
raw_path = RUN_DIR / "requests.jsonl"
if raw_path.exists():
    raise RuntimeError("Requests already exist for this run; start a new run instead of appending duplicate evidence.")
for concurrency in CONCURRENCIES:
    for repeat in range(REPEATS):
        batch_start = time.perf_counter()
        ordered = [None] * len(cases)
        with ThreadPoolExecutor(max_workers=concurrency) as pool:
            futures = {pool.submit(call_case, case): i for i, case in enumerate(cases)}
            for future in as_completed(futures):
                index = futures[future]
                result = future.result()
                result.update(concurrency=concurrency, repeat=repeat)
                ordered[index] = result
                # Persist each completed request immediately.
                with raw_path.open("a", encoding="utf-8") as fh:
                    fh.write(json.dumps(result, ensure_ascii=False, default=str) + "\n")
        elapsed = time.perf_counter() - batch_start
        records.extend(ordered)
        total_tokens = sum(r["completion_tokens"] or 0 for r in ordered)
        ok_count = sum(r["transport_ok"] and r["finish_reason"] == "stop" for r in ordered)
        batch_metrics.append(dict(concurrency=concurrency, repeat=repeat,
            wall_s=elapsed, requests=len(ordered), completed_requests=ok_count,
            completion_tokens=total_tokens, requests_per_s=ok_count / elapsed,
            aggregate_completion_tokens_per_s=total_tokens / elapsed))
        print(f"concurrency={concurrency}, pass={repeat + 1}, complete={ok_count}/{len(cases)}, "
              f"wall={elapsed:.1f}s, aggregate output={total_tokens / elapsed:.1f} tokens/s")
save_json("batch_metrics.json", batch_metrics)


concurrency=1, pass=1, complete=29/29, wall=49.8s, aggregate output=63.3 tokens/s
concurrency=1, pass=2, complete=29/29, wall=49.1s, aggregate output=64.1 tokens/s
concurrency=4, pass=1, complete=29/29, wall=14.9s, aggregate output=210.2 tokens/s
concurrency=4, pass=2, complete=29/29, wall=13.8s, aggregate output=227.0 tokens/s


PosixPath('/content/datamate_evaluation/20261109T040332_785805Z/batch_metrics.json')

## 11. Tables and acceptance summary

PASS means only that this run met the named check. Missing evidence remains **NOT VERIFIED**. The optional fixture score uses gold/reference frames, not the candidate parser, and must not be reported as competition PA.

A100 timing thresholds below are project diagnostic targets (SPEC §11), not a pass for G3/G6. The GPU sampler can miss short-lived allocation spikes.


In [15]:
def percentile(values, p):
    values = sorted(values)
    if not values:
        return None
    rank = (len(values) - 1) * p / 100
    low, high = math.floor(rank), math.ceil(rank)
    return values[low] + (values[high] - values[low]) * (rank - low)

def show_table(rows):
    if not rows:
        display(Markdown("_No rows._"))
        return
    keys = list(rows[0])
    def clean(v):
        return str(v).replace("|", "/").replace("\n", " ")
    text = "| " + " | ".join(keys) + " |\n| " + " | ".join("---" for _ in keys) + " |\n"
    text += "\n".join("| " + " | ".join(clean(row.get(k, "")) for k in keys) + " |" for row in rows)
    display(Markdown(text))

performance_rows = []
for concurrency in CONCURRENCIES:
    for stage in ("parse", "solve", "repair"):
        group = [r for r in records if r["concurrency"] == concurrency and r["stage"] == stage]
        successful = [r for r in group if r["transport_ok"] and r["finish_reason"] == "stop"]
        latencies = [r["elapsed_s"] for r in successful]
        performance_rows.append(dict(concurrency=concurrency, stage=stage, n=len(group),
            completed=len(successful), schema_valid=sum(r["schema_valid"] for r in group),
            app_valid=sum(r["app_valid"] for r in group),
            p50_s=round(percentile(latencies, 50), 3) if latencies else None,
            p95_s=round(percentile(latencies, 95), 3) if latencies else None,
            output_tokens=sum(r["completion_tokens"] or 0 for r in successful)))
show_table(performance_rows)

determinism_rows = []
for concurrency in CONCURRENCIES:
    for case in cases:
        group = sorted([r for r in records if r["concurrency"] == concurrency
                        and r["case_id"] == case["case_id"]], key=lambda r: r["repeat"])
        complete = len(group) == REPEATS and all(
            r["transport_ok"] and r["schema_valid"] and r["finish_reason"] == "stop" for r in group)
        same = complete and all(
            (r["content"], r["reasoning"]) == (group[0]["content"], group[0]["reasoning"])
            for r in group[1:])
        determinism_rows.append(dict(concurrency=concurrency, case_id=case["case_id"],
                                     complete=complete, exact_match=same))
show_table([dict(concurrency=c,
    matches=sum(r["exact_match"] for r in determinism_rows if r["concurrency"] == c),
    total=len(cases)) for c in CONCURRENCIES])
cross_concurrency = []
for case in cases:
    group = [r for r in records if r["case_id"] == case["case_id"]]
    good = all(r["transport_ok"] and r["schema_valid"] and r["finish_reason"] == "stop" for r in group)
    cross_concurrency.append(dict(case_id=case["case_id"], exact_match=good and all(
        (r["content"], r["reasoning"]) == (group[0]["content"], group[0]["reasoning"]) for r in group)))
peak_mib = max((s["used_mib"] for s in gpu_samples if "used_mib" in s), default=None)
def verdict(condition):
    return "PASS" if condition else "FAIL"
compliance_reviewed = LICENSE_REVIEWED_AND_PERMITTED and bool(RELEASE_EVIDENCE_URL.strip())
sql_rows = [r for r in records if r["stage"] != "parse" and not r["diagnostic"]]
acceptance = [
    dict(criterion="Stored BF16 checkpoint tensors <= 15B", status=verdict(MODEL_EVIDENCE["parameter_limit_pass"]),
         evidence=str(tensor_count)),
    dict(criterion="License + original weights release review",
         status="REVIEWED — organizer interpretation still applies" if compliance_reviewed else "NOT VERIFIED",
         evidence=RELEASE_EVIDENCE_URL or "Fill evidence and review license"),
    dict(criterion="A100/local serving startup", status="PASS", evidence=f"{LOAD_SECONDS:.2f}s"),
    dict(criterion="HTTP + schema + parser invariants + complete output",
         status=verdict(all(r["transport_ok"] and r["schema_valid"] and r["app_valid"]
                            and r["finish_reason"] == "stop" for r in records)),
         evidence=f"{len(records)} requests, errors preserved in requests.jsonl"),
    dict(criterion="Isolated fixture SQL execution + typed result",
         status=verdict(bool(sql_rows) and all(r["sql_ok"] and r["typed_ok"] for r in sql_rows))
                if EXECUTE_FIXTURE_SQL else "NOT VERIFIED",
         evidence="Reference frames; final fixture scores reported separately; not full-pipeline PA"),
    dict(criterion="Exact repeatability within each tested concurrency",
         status=verdict(all(r["exact_match"] for r in determinism_rows)),
         evidence=f"{sum(r['exact_match'] for r in determinism_rows)}/{len(determinism_rows)}"),
    dict(criterion="Cross-concurrency exact output (diagnostic)",
         status=verdict(all(r["exact_match"] for r in cross_concurrency)),
         evidence=f"{sum(r['exact_match'] for r in cross_concurrency)}/{len(cases)}"),
    dict(criterion="Sampled total GPU memory <= 32 GiB (proxy)",
         status=verdict(peak_mib <= 32 * 1024) if peak_mib is not None else "NOT VERIFIED",
         evidence=f"sampled peak={peak_mib} MiB; engine budget={TARGET_EXECUTOR_GIB} GiB"),
    dict(criterion="sm_120 / RTX 5090 / Docker / full-run runtime",
         status="NOT VERIFIED", evidence="Repeat on E5/G4 and official container; A100 is insufficient"),
    dict(criterion="Training/held-out PA and final model choice",
         status="NOT VERIFIED", evidence="M3-08 / Lead harness required"),
    dict(criterion="M3-01 comparison of 2–3 candidates",
         status="NOT VERIFIED", evidence="This notebook evaluates Qwen3-8B only"),
]
show_table(acceptance)
fixture_rows = [r for r in records if r.get("fixture_score") is not None]
print("Isolated solver/repair fixture correct:", sum(r["fixture_score"] == 1 for r in fixture_rows),
      "/", len(fixture_rows), "(reference frames; repeated requests; NOT competition PA)")
parser_type_rows = [r for r in records if "expected_type_correct" in r]
print("Parser expected_type on answerable fixture references:",
      sum(r["expected_type_correct"] for r in parser_type_rows), "/", len(parser_type_rows))
save_json("performance.json", performance_rows)
save_json("determinism.json", determinism_rows)
save_json("cross_concurrency.json", cross_concurrency)
save_json("acceptance.json", acceptance)
for filename, rows in [("performance.csv", performance_rows), ("acceptance.csv", acceptance),
                       ("batches.csv", batch_metrics), ("determinism.csv", determinism_rows)]:
    with (RUN_DIR / filename).open("w", newline="", encoding="utf-8-sig") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


| concurrency | stage | n | completed | schema_valid | app_valid | p50_s | p95_s | output_tokens |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | parse | 32 | 32 | 32 | 32 | 1.489 | 2.371 | 2676 |
| 1 | solve | 24 | 24 | 24 | 24 | 2.317 | 2.782 | 3399 |
| 1 | repair | 2 | 2 | 2 | 2 | 1.705 | 1.713 | 220 |
| 4 | parse | 32 | 32 | 32 | 32 | 1.669 | 2.613 | 2670 |
| 4 | solve | 24 | 24 | 24 | 24 | 2.532 | 3.103 | 3380 |
| 4 | repair | 2 | 2 | 2 | 2 | 1.877 | 1.883 | 220 |

| concurrency | matches | total |
| --- | --- | --- |
| 1 | 26 | 29 |
| 4 | 23 | 29 |

| criterion | status | evidence |
| --- | --- | --- |
| Stored BF16 checkpoint tensors <= 15B | PASS | 8190735360 |
| License + original weights release review | NOT VERIFIED | Fill evidence and review license |
| A100/local serving startup | PASS | 120.09s |
| HTTP + schema + parser invariants + complete output | PASS | 116 requests, errors preserved in requests.jsonl |
| Isolated fixture SQL execution + typed result | FAIL | Reference frames; final fixture scores reported separately; not full-pipeline PA |
| Exact repeatability within each tested concurrency | FAIL | 49/58 |
| Cross-concurrency exact output (diagnostic) | FAIL | 22/29 |
| Sampled total GPU memory <= 32 GiB (proxy) | PASS | sampled peak=29840.0 MiB; engine budget=28.8 GiB |
| sm_120 / RTX 5090 / Docker / full-run runtime | NOT VERIFIED | Repeat on E5/G4 and official container; A100 is insufficient |
| Training/held-out PA and final model choice | NOT VERIFIED | M3-08 / Lead harness required |
| M3-01 comparison of 2–3 candidates | NOT VERIFIED | This notebook evaluates Qwen3-8B only |

Isolated solver/repair fixture correct: 24 / 32 (reference frames; repeated requests; NOT competition PA)
Parser expected_type on answerable fixture references: 44 / 44


For the 3 failed criteria, we run a diagnosis test to determine what could cause the issue.

In [18]:
from collections import defaultdict

print("=== SQL / TYPED RESULT FAILURES ===")
for r in records:
    if r["stage"] == "parse" or r["diagnostic"]:
        continue
    if r.get("sql_ok") and r.get("typed_ok"):
        continue

    print(
        "\n", r["case_id"],
        "concurrency=", r["concurrency"],
        "repeat=", r["repeat"],
    )
    for key in [
        "http_status", "finish_reason", "error",
        "validation_error", "sql_error", "sql_detail", "type_error",
    ]:
        if r.get(key) is not None:
            print(key, ":", r[key])

    execution = r.get("execution") or {}
    print("execution_error:", execution.get("error_text"))
    print("content:", (r.get("content") or "")[:1200])


print("\n=== REPEATABILITY FAILURE CLASSIFICATION ===")
groups = defaultdict(list)
for r in records:
    groups[(r["case_id"], r["concurrency"])].append(r)

for (case_id, concurrency), group in groups.items():
    group.sort(key=lambda r: r["repeat"])

    complete = len(group) == REPEATS and all(
        r["transport_ok"]
        and r["schema_valid"]
        and r["finish_reason"] == "stop"
        for r in group
    )

    if not complete:
        print(case_id, concurrency, "INCOMPLETE_OR_INVALID")
        continue

    exact = len({
        (r["content"], r["reasoning"]) for r in group
    }) == 1

    if not exact:
        print(case_id, concurrency, "VALID_BUT_DIFFERENT")
        for r in group:
            print(" repeat:", r["repeat"])
            print(" content:", (r["content"] or "")[:1200])

=== SQL / TYPED RESULT FAILURES ===

 solve:fx_q04 concurrency= 1 repeat= 0
http_status : 200
finish_reason : stop
type_error : E_TYPE_CONTRACT
execution_error: None
content: {
  "sql": "SELECT ten_hang, SUM(so_luong) AS total_quantity, ROW_NUMBER() OVER (ORDER BY SUM(so_luong) DESC, ma_hang ASC) AS sort_key FROM ban_hang WHERE ngay_dat BETWEEN '2026-03-01' AND '2026-03-04' GROUP BY ten_hang, ma_hang HAVING SUM(so_luong) > 0 ORDER BY total_quantity DESC, ma_hang ASC LIMIT 3",
  "notes": "The query calculates the total quantity sold for each item in the resolved month window (March 1 to March 4, 2026). It then ranks the items by total quantity sold in descending order, and in case of a tie, by item code in ascending order. The top 3 items are returned."
}

 solve:fx_q05 concurrency= 1 repeat= 0
http_status : 200
finish_reason : stop
sql_error : E_SQL_EXEC
sql_detail : ParserException: Parser Error: syntax error at or near ","
execution_error: None
content: {
  "sql": "SELECT SUM(thanh_t

## 12. Stop serving and preserve the run

Always run this cell after the experiment, including after a failed benchmark. It stops only the server process launched by this notebook. It preserves downloaded weights and all run artifacts.

No automatic full-run pass is assigned. Fill the conclusion below after reviewing failed cases and the exported tables.


In [16]:
if "server" in globals() and server.poll() is None:
    os.killpg(server.pid, signal.SIGTERM)
    try:
        server.wait(timeout=30)
    except subprocess.TimeoutExpired:
        os.killpg(server.pid, signal.SIGKILL)
        server.wait(timeout=10)
if "monitor_stop" in globals():
    monitor_stop.set()
    monitor_thread.join(timeout=3)
if "server_log_handle" in globals():
    server_log_handle.close()
save_json("gpu_samples.json", globals().get("gpu_samples", []))
manifest = dict(
    notebook="MODEL_EVALUATION.ipynb", task="M3-01", repo_commit=REPO_COMMIT,
    contracts_version=globals().get("CONTRACTS_VERSION"), templates=template_sha256(),
    schema_adapter=SCHEMA_ADAPTER_POLICY,
    schema_evidence_file="schema_evidence.json",
    gpu=GPUS, serving_environment=SERVER_ENV, model=MODEL_EVIDENCE,
    max_model_len=MAX_MODEL_LEN, max_num_seqs=MAX_NUM_SEQS,
    tested_concurrencies=CONCURRENCIES, repeats=REPEATS, seed=SEED,
    target_executor_gib=TARGET_EXECUTOR_GIB,
    actual_memory_fraction=globals().get("GPU_UTILIZATION"),
    inference_network_isolation="HF/Transformers offline flags only; NOT OS-isolated",
    inference_backend="direct local HTTP; no changes to production ModelClient",
)
save_json("run_manifest.json", manifest)
report = """# M3-01 review — fill after inspecting the run

- Preliminary A100 serving conclusion: [PASS / FAIL / NEEDS INVESTIGATION]
- Model revision: {revision}
- Evidence directory: {directory}
- Failed cases and likely causes: [fill]
- Chosen context/concurrency and memory margin: [fill]
- Schema/invariant failures: [fill]
- Repeatability failures, including cross-concurrency differences: [fill]
- Throughput/latency interpretation and cache conditions: [fill]
- License/release/parameter review owner and decision: [fill]
- Recommendation for M3-02 and M4-03: [fill]
- Next candidates to compare: [fill]
- E5/sm_120/Docker validation: NOT VERIFIED by this A100 run.
- Full training / held-out PA: NOT VERIFIED; fixture checks are not competition accuracy.
- Lead/M4 review: [fill]
"""
(RUN_DIR / "CONCLUSION.md").write_text(report.format(
    revision=revision, directory=RUN_DIR), encoding="utf-8")
print("Artifacts preserved in:", RUN_DIR)
print("Before ending Colab, download the ZIP in the next cell or copy this directory to Drive.")


Artifacts preserved in: /content/datamate_evaluation/20261109T040332_785805Z
Before ending Colab, download the ZIP in the next cell or copy this directory to Drive.


## 13. Download results (optional)

The ZIP contains requests/responses, tables, logs, package versions, checkpoint evidence and a conclusion template. It excludes model weights and temporary DuckDB files. Colab storage is ephemeral. Review artifact contents before sharing them.


In [17]:
# import zipfile
# archive_path = EVAL_ROOT / f"{RUN_STAMP}.zip"
# with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
#     for path in sorted(RUN_DIR.rglob("*")):
#         if path.is_file() and "work" not in path.relative_to(RUN_DIR).parts:
#             archive.write(path, arcname=str(path.relative_to(RUN_DIR)))
# print(archive_path)
# try:
#     from google.colab import files
# except ImportError:
#     print("Outside Colab: download the archive using your file browser.")
# else:
#     files.download(str(archive_path))


/content/datamate_evaluation/20261109T040332_785805Z.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Troubleshooting and remaining work

- **No GPU / too little free memory:** select an A100 runtime; stop unrelated GPU processes. Never increase the budget beyond the target to hide an OOM.
- **Dependency or CUDA startup failure:** inspect `server.log` and the serving lock. Recreate a fresh runtime/venv for a changed stack; do not mix versions from separate experiments.
- **Schema compilation/API failure:** inspect the exact HTTP response. Verify support for the actual Pydantic schema and pinned vLLM version; do not silently fall back to unconstrained text generation.
- **Output truncated:** inspect `finish_reason`. Increase the stage budget only with a corresponding context check and a new run.
- **Warm-up or benchmark failure:** run cleanup, preserve logs, then investigate. Do not report missing rows as successful.
- **Timing interpretation:** model load and initial schema warm-up are recorded separately; benchmark latency is warm-cache API latency. No TTFT is claimed from non-streaming requests.
- **M3-01 completion:** compare 2–3 eligible candidates, review M4's lock, and validate compatibility on sm_120. Do not infer it from A100.
- **M3-08:** run the real full pipeline with the Lead's business-fold harness; evaluate final answers with the scorer and report penalized accuracy, runtime, and determinism.
- **Notebook validation:** code structure can be checked without GPUs; this file ships without fabricated execution outputs. A successful Colab run must supply the actual evidence.

Official references:
1. [Qwen3-8B model card](https://huggingface.co/Qwen/Qwen3-8B)
2. [Qwen3 release announcement](https://qwenlm.github.io/blog/qwen3/)
3. [Qwen3-8B license](https://huggingface.co/Qwen/Qwen3-8B/blob/main/LICENSE)
4. [vLLM 0.10.1.1 GPU installation](https://docs.vllm.ai/en/v0.10.1.1/getting_started/installation/gpu.html)
5. [vLLM 0.10.1.1 structured outputs](https://docs.vllm.ai/en/v0.10.1.1/features/structured_outputs.html)
6. [vLLM 0.10.1.1 engine arguments](https://docs.vllm.ai/en/v0.10.1.1/configuration/engine_args.html)
